# Datamine VGM3DMAP Process Example

This notebook demonstrates how to configure and run the **`vgm3dmap`** process wrapper in `dmstudio`.

## Process Description

## VGM3DMAP

VGM3DMAP is used to interpolate experimental variogram values into a 3D map (represented by a block model).

It is used during the [Investigate Anisotropy](<../STUDIO_RM/Multivariate_Investigate_Anisotropy.md>) phase of Advanced Estimation, although can also be used as a standalone process to output a 3D map block model (3DMAP) and an optional file containing the experimental variograms used to generate the map.

The variogram block model is in standard Datamine block model format. It consists of a 3D matrix of cubic parent cells with the centre of the model assigned the coordinates of 0, 0, 0. The variogram values that are estimated into the model show how the variance increases (correlation decreases) as a function of distance and orientation from the centre point. Viewing planar slices through the centre point allows anisotropy to be identified showing where the correlation between samples differs in different directions. This analysis helps to identify the directions of the axes of the model variogram to be fitted.

### 3D Variogram Block Model

The &3DMAP file will contain the interpolated output variogram block model values It will include a field for the variogram value for each grade as specified by the *Gi fields.

The model file also includes a field showing the weighted average number of sample pairs used to calculate each variogram value. This is calculated by interpolating the number of sample pairs used to create each grade variogram value using the same inverse distance method and parameters as for the grade variogram. The name of this field in the output model is created by combining the grade field name followed by the characters "_N". Applying a legend to this value will show the spatial distribution of the density of information on model sections.

The model can be viewed in the 3D window using all the standard model display options. However it is not currently possible to load it into the Variograms window to view orthogonal sections through the model.

There is a choice of methods to generate the data used for interpolation to create the 3D variogram model. If GRIDMODE=0 then standard angular experimental variograms are calculated radiating from a centre point at different azimuths and dips. Each lag point along every variogram vector has two data items the variogram value and the number of sample pairs.

If GRIDMODE=1 then the volume around the centre point is divided into a 3D set of cubes. The size of the cubes is defined by the lag distance. Every pair of samples is assigned to one of the cubes based on the azimuth, dip and length of the joining vector. The weighted average variogram value for each cell is assigned to a location within the cell, calculated as a weighted average position of the sample vectors within it. In this way the average variogram value and the number of samples can be calculated for each cell.

The difference between the two GRIDMODE settings is therefore the spatial location of the variogram values and the sample number data. Whichever method is selected these two data items are interpolated into the 3D variogram model using the inverse power of distance method. GRIDMODE=1 is applied when the Investigate Anisotropy option is run from the Advanced Estimation screen.

### Input Files:

* **samples** (Undefined):
  A Datamine file that contains sample positional information and supporting attributes.
  Required=Yes

### Output Files:

* **vgram** (Undefined):
  Optional output experimental variograms used to create the 3D interpolated variogram.
  Requires GRIDMODE = 0.
  Required=No

### Fields:

* **gs** (Undefined : Undefined):
  Grade fields for variogram calculation. At least G1 must be specified.
  Default=Undefined
  Required=No

### Parameters:

* **nblocks**:
  The number of blocks along each axis for the output variogram block model. A greater
  number of blocks will result in a finer grid but will require longer calculation times.
  Range=Undefined
  Values=Undefined
  Default=40
  Required=Yes

* **maxlag**:
  The maximum distance between pairs of samples to be considered for variogram
  calculation.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=Yes

* **nlags**:
  The number of lags used to calculate experimental variograms. It is recommended that
  this value is less than **NBLOCKS**. The incremental lag distance is then calculated as

## **MAXLAG** / **NLAGS**.

  Range=1,200
  Values=Undefined
  Default=25
  Required=Yes

* **nangles**:
  The number of angles (directions) for experimental variogram calculation. The increment
  between directions for both azimuth and dip is then calculated as 180 / **NANGLES**
  degrees. A greater number may allow more structure to be revealed in the resulting
  variogram map. This is not used for **GRIDMODE** =1.
  Range=1,180
  Values=Undefined
  Default=9
  Required=No

* **gridmode**:
  If set to 1 a regular grid is used to calculate experimental variogram values to be
  interpolated into the output variogram map. If set to 0 standard directional variograms
  are used.
  Range=0,1
  Values=0,1
  Default=0
  Required=Yes

* **disc**:
  The number of discretization points in X, Y and Z used for inverse distance
  interpolation of the variogram map.
  Range=1,10
  Values=Undefined
  Default=3
  Required=No

* **minsamp**:
  The minimum number of experimental variogram points to be found within the search
  radius when performing inverse distance interpolation of the variogram block model.
  Range=1,10
  Values=Undefined
  Default=1
  Required=No

* **optsamp**:
  The optimum number of experimental variogram points to be found within the search
  radius when performing inverse distance interpolation of the variogram map.
  Range=1,100
  Values=Undefined
  Default=50
  Required=No

* **searchlf**:
  The radius of the initial search sphere for each interpolated variogram map point as a
  factor of the lag distance.
  Range=0,100
  Values=Undefined
  Default=2
  Required=No

* **search2f**:
  The factor of the search sphere relative to the second search sphere.
  Range=0,100
  Values=Undefined
  Default=1
  Required=No

* **search3f**:
  The factor of the search sphere relative to the second search sphere.
  Range=0,100
  Values=Undefined
  Default=1
  Required=No

* **invdistp**:
  The inverse distance power used when performing inverse distance interpolation of the
  variogram map.
  Range=0,100
  Values=Undefined
  Default=1
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob

import pandas as pd

from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('vgm3dmap')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.copy_database_files(files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute vgm3dmap
print("Running vgm3dmap...")
dm_cmd.vgm3dmap(
    samples_i=['t_assays'],  # required
    gs_f=['optional'],  # required
    nblocks_p='required_val',  # required
    maxlag_p='required_val',  # required
    nlags_p='required_val',  # required
    gridmode_p='required_val',  # required
    # vgram_o='t_vgm3dmap_out',  # optional
    # nangles_p=9,  # optional
    # disc_p=3,  # optional
    # minsamp_p=1,  # optional
    # optsamp_p=50,  # optional
    # searchlf_p=2,  # optional
    # search2f_p=1,  # optional
    # search3f_p=1,  # optional
    # invdistp_p=1,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("vgm3dmap execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_vgm3dmap_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")